# COMP9085 Week 11 Laboratory
## Boosting, pretrained protein embeddings and fine-tuning, on one biological question

**Two hours. No GPU.** The lecture followed one question from start to finish: *does this human protein carry an
N-terminal signal peptide*, the short hydrophobic "address label" that sends a protein into the secretory pathway?
Today you answer it four ways, under one protocol: designed sequence descriptors with a single tree, a forest and
**gradient boosting**; **frozen ESM-2 embeddings** with a simple classifier; and, from recorded runs, **LoRA** and
**full fine-tuning** of ESM-2. Then you decide what you would actually recommend.

**Data** (`data/`, see `data/README.md`): the Week 7 proteins: 2,000 reviewed human UniProtKB/Swiss-Prot proteins
(release 2026_03), one per UniRef50 cluster (no two share ≥ 50% sequence identity).
* `swissprot_human_2000.csv`: sequences, UniProt keywords, the evidence behind each signal-peptide annotation;
* `proteins_classical_features.csv`: the 23 Week 7 descriptors (20 amino-acid fractions, log length, maximum
  hydropathy in a 19-residue window, mean hydropathy of the first 25 residues);
* `esm2_t6_8M_embeddings.csv.gz`: ESM-2 (8M) final layer, averaged over the **whole** sequence (320 numbers);
* `esm2_t6_8M_nterm100_embeddings.csv.gz`: the same model, averaged over the **first 100 residues** only;
* `week11_finetune_results.json`: the recorded LoRA and full fine-tuning runs (validation curves, settings, test
  probabilities, timings). They were run with `COMP9085_Week11_FineTuning_Demo.ipynb` settings on a laptop CPU.

**Protocol.** The Week 7 split: 1,500 training and 500 test proteins. For Part C the training proteins are split
again into 1,200 *fit* and 300 *validation* proteins. **The 500 test proteins are scored once per pipeline, after
every choice has been made.** Requirements: Python 3.10+, numpy, pandas, matplotlib, scikit-learn ≥ 1.3. Runs in
about three to five minutes on a laptop.

Week 11 page: https://hafli.github.io/comp9085-ml-biology/weeks/week11.html

In [1]:
# provided: setup
import json, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold, train_test_split, GridSearchCV,
                                     cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings("ignore")
SEED = 42

URL = "https://hafli.github.io/comp9085-ml-biology/labs/week11/data/"
FILES = ["swissprot_human_2000.csv", "proteins_classical_features.csv", "esm2_t6_8M_embeddings.csv.gz",
         "esm2_t6_8M_nterm100_embeddings.csv.gz", "week11_finetune_results.json"]
DATA = next((q for q in [Path("data"), Path("../data")] if (q / FILES[0]).exists()), None)
if DATA is None:                                   # Google Colab: fetch the files from the course website
    import urllib.request
    DATA = Path("data"); DATA.mkdir(exist_ok=True)
    for f in FILES:
        urllib.request.urlretrieve(URL + f, DATA / f)

seqs = pd.read_csv(DATA / "swissprot_human_2000.csv")
prot = pd.read_csv(DATA / "proteins_classical_features.csv")
assert (seqs.accession == prot.accession).all()
FEATURES = [c for c in prot.columns if c.startswith("frac_")] + ["log_length", "max_hydropathy_19", "nterm_hydropathy_25"]
X = prot[FEATURES].values
E_whole = pd.read_csv(DATA / "esm2_t6_8M_embeddings.csv.gz").set_index("accession").loc[prot.accession].values
E_nterm = pd.read_csv(DATA / "esm2_t6_8M_nterm100_embeddings.csv.gz").set_index("accession").loc[prot.accession].values
FT = json.load(open(DATA / "week11_finetune_results.json"))
y, tm = prot.has_signal_peptide.values, prot.is_transmembrane.values

idx = np.arange(len(prot))
tr, te = train_test_split(idx, test_size=0.25, stratify=2 * tm + y, random_state=SEED)      # the Week 7 split
fit, val = train_test_split(tr, test_size=0.2, stratify=y[tr], random_state=SEED)           # Part C only

def cv5():
    return StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def logreg(C=1.0):
    return Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(C=C, max_iter=5000, random_state=SEED))])

print(len(prot), "proteins;", len(FEATURES), "descriptors;", E_whole.shape[1], "embedding dimensions;",
      len(tr), "train (", len(fit), "fit +", len(val), "validation ) /", len(te), "test")

2000 proteins; 23 descriptors; 320 embedding dimensions; 1500 train ( 1200 fit + 300 validation ) / 500 test


# Part A. The problem and the baselines

## Task 1. Look at the data before modelling it (10 min)

**1a.** Report the number and share of proteins with a signal peptide overall, in the training set and in the test set.

**1b.** Tabulate `seqs.sp_evidence` for the proteins **with** a signal peptide. How many labels rest on experimental
evidence, and how many on "sequence analysis" (agreement between prediction tools such as SignalP and Phobius)?

**1c.** Print the first 30 residues of five proteins with a signal peptide and five without, and compare the median of
`nterm_hydropathy_25` (mean Kyte–Doolittle hydropathy of the first 25 residues) between the two groups.

In [2]:
# your code here


**Question 1.** What exactly is the label you are predicting? Given 1b, what would a perfect score on this test set prove, and what would it not prove?

*Your answer here.*

## Task 2. Baselines first (10 min)

With `cv5()` on the **1,500 training proteins**, report the mean and SD of ROC AUC for
1. `DummyClassifier(strategy="prior")` (always the base rate);
2. `logreg(C=0.1)`;
3. `DecisionTreeClassifier(max_depth=4, random_state=SEED)`;
4. `RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)`.

Write `cv_auc(model, M)` that returns the five fold AUCs for features `M[tr]` and labels `y[tr]` (use `cross_val_score`).

In [3]:
def cv_auc(model, M):
    # your code here
    pass

# your code here

# Part B. Boosting

## Task 3. Learning rate and the number of trees (20 min)

Gradient boosting adds trees one at a time, each fitted to the errors (the negative gradient of the loss) of the
ensemble so far, scaled by the **learning rate** ν: F_m(x) = F_{m−1}(x) + ν·h_m(x).

Fit `GradientBoostingClassifier(learning_rate=lr, n_estimators=400, max_depth=3, random_state=SEED)` on the
**fit** proteins for lr in 1.0, 0.3, 0.1 and 0.03. With `staged_predict_proba`, compute the log-loss on the fit
proteins and on the **validation** proteins after every tree. Plot both against the number of trees (log x-axis) and
report, for each rate, the number of trees with the lowest validation log-loss.

In [4]:
# your code here


**Question 3.** How do the learning rate and the best number of trees trade off? Why does the validation curve turn upwards for large rates while the training curve keeps falling, and why could a random forest not show this?

*Your answer here.*

## Task 4. Depth, subsampling and early stopping (15 min)

**4a.** Repeat Task 3 at learning rate 0.1 for `max_depth` 1, 3 and 6, and for `subsample` 1.0 and 0.5
(stochastic gradient boosting: each tree sees a random half of the rows). Report best trees and best validation log-loss.

**4b.** `HistGradientBoostingClassifier` is scikit-learn's histogram-based implementation, the same family of algorithm
as XGBoost and LightGBM. Fit it on the **training** proteins with `early_stopping=True, validation_fraction=0.2,
n_iter_no_change=20, max_iter=1000, learning_rate=0.1`. How many iterations did it keep (`n_iter_`)?

In [5]:
# your code here


**Question 4.** What do deeper trees and subsampling each do to the bias–variance balance? Early stopping set aside 20% of the training proteins: why is that legitimate, while stopping on the test set would not be?

*Your answer here.*

## Task 5. Choose the settings without touching the test set (20 min)

**5a.** Tune `GradientBoostingClassifier(random_state=SEED)` with `GridSearchCV` (`cv=cv5()`, `scoring="roc_auc"`) on
the **training** proteins over `learning_rate` ∈ {0.03, 0.1, 0.3}, `n_estimators` ∈ {100, 300}, `max_depth` ∈ {2, 3}
and `subsample` ∈ {0.5, 1.0}. Report the chosen settings and the CV AUC.

**5b.** Only now, score the **test** proteins once with the refitted best model, and also with the logistic regression,
the single tree and the random forest of Task 2 (each fitted on all 1,500 training proteins). Report test ROC AUC and
average precision in one table.

In [6]:
# your code here


**Question 5.** Is the tuned boosted model clearly better than the forest and the logistic regression on these 23 descriptors? What would you need before claiming that it is? Why is the CV AUC of the chosen setting itself slightly optimistic?

*Your answer here.*

## Task 6. What does the boosted model rely on? (10 min)

Compute the impurity importances (`gb_best.feature_importances_`) and the permutation importances on the **test**
proteins (`permutation_importance(..., scoring="roc_auc", n_repeats=20, random_state=SEED)`). Show the top 8 of each.

In [7]:
# your code here


**Question 6.** The top feature agrees with textbook biology. Does that make the model's importance ranking an explanation of how signal peptides work? Give two reasons it might rank a feature highly that is not causally involved.

*Your answer here.*

## Task 7. Related proteins across the split (10 min)

The proteins were chosen one per UniRef50 cluster, but members of one gene family (olfactory receptors `OR…`,
solute carriers `SLC…`, zinc fingers `ZNF…`) are still distant relatives. Define a family for each training protein
from its gene symbol (provided) and compare 5-fold CV AUC with `cv5()` against `StratifiedGroupKFold(5, shuffle=True,
random_state=SEED)` with `groups=family[tr]`, for `gb_best` on the descriptors and for `logreg(0.01)` on `E_whole`.

In [8]:
def family(gene, acc):
    """Leading letters of the gene symbol (OR2A4 -> OR); no symbol or a chromosome ORF -> its own group."""
    if not isinstance(gene, str) or not gene:
        return acc
    if "orf" in gene:
        return gene
    m = re.match(r"[A-Za-z]+", gene)
    return m.group(0).upper() if m else gene

family = np.array([family(g, a) for g, a in zip(prot.gene, prot.accession)])
print(len(set(family[tr])), "families among the training proteins; largest:",
      pd.Series(family[tr]).value_counts().head(6).to_dict())

1048 families among the training proteins; largest: {'ZNF': 40, 'SLC': 35, 'TMEM': 17, 'OR': 14, 'FAM': 13, 'CCDC': 10}


In [9]:
# your code here


**Question 7.** Did keeping families together change the estimate? Why might it matter much more for a dataset that was *not* reduced to one protein per UniRef50 cluster, and what is the extra issue for ESM-2 (pretraining data)?

*Your answer here.*

# Part C. Pretrained protein representations: the adaptation ladder

From here on every pipeline uses the same three sets: fit on the **1,200 fit** proteins, choose settings on the
**300 validation** proteins, score the **500 test** proteins once. The fine-tuning runs in `FT` used exactly this
protocol, so all rungs are comparable.

## Task 8. Frozen embeddings and a simple head (15 min)

Write `frozen_head(M)` that fits `logreg(C)` on `M[fit]` for C ∈ {0.001, 0.01, 0.1, 1.0}, picks the C with the
best validation ROC AUC, and returns that C, the validation AUC, the test ROC AUC and the test probabilities.
Apply it to `E_whole`, `E_nterm` and the descriptors `X`. Also fit `HistGradientBoostingClassifier(random_state=SEED,
max_iter=300, learning_rate=0.1, max_leaf_nodes=7)` on `E_nterm[fit]` and report its test AUC.

In [10]:
def frozen_head(M, Cs=(0.001, 0.01, 0.1, 1.0)):
    # your code here
    pass

# your code here

**Question 8.** The model and its weights are identical in the two ESM-2 rows; only the residues averaged differ. Why does that matter so much for this task, and what does it tell you about 'using embeddings' as a single option?

*Your answer here.*

## Task 9. Fine-tuning: read the recorded runs (15 min)

Fine-tuning updates the pretrained network itself on the labelled proteins. `FT["finetune"]["runs"]` holds two
methods, each tuned on validation (learning rate, epoch) and repeated with three seeds:
* `"lora"`: **LoRA**: the pretrained weights stay frozen; rank-8 matrices B·A are learned beside the query and value
  projections of every layer, plus the classification head;
* `"full"`: **full fine-tuning**: every weight is updated.

**9a.** For each method, print the chosen learning rate, trainable parameters, total parameters, training seconds per
epoch and, per seed, the best epoch, validation AUC and test AUC.

**9b.** Plot the validation AUC against epoch for seed 0 of both methods (`["curve"]`).

**9c.** Bootstrap (2,000 resamples of the 500 test proteins, `rng = np.random.default_rng(SEED)`) the difference in
test AUC between LoRA (seed 0, `test_prob`) and the frozen first-100 head of Task 8. Report the 95% interval.

**9d.** Write `lora_params(r, d, layers, matrices_per_layer=2)` = r·(d + d)·matrices·layers and check that, with the
ESM-2 8M sizes (d = 320, 6 layers, r = 8) plus the classification head (320·320 + 320 + 320·2 + 2), it equals the
recorded trainable count.

In [11]:
R = FT["finetune"]["runs"]
# your code here

**Question 9.** Does fine-tuning beat the frozen first-100 head convincingly on this task? Weigh the size of the gain, its interval and the seed-to-seed spread against the cost (trainable parameters, time, software, and the extra choices that each need validation).

*Your answer here.*

## Task 10. The recommendation (10 min)

Fill in the table (one row per rung: descriptors + boosting, frozen whole, frozen first 100, LoRA, full fine-tuning)
with test AUC, trainable parameters, approximate compute, software needed and interpretability. Then write five
sentences to a collaborator who has **400 labelled proteins of a new organism** and asks which approach to use.

In [12]:
# your code here

*Your answer here.*

## Optional extension. Run a fine-tuning yourself

`COMP9085_Week11_FineTuning_Demo.ipynb` reproduces one LoRA run of the recorded results on a laptop CPU (about five
minutes; it downloads the 30 MB ESM-2 8M model and needs `torch`, `transformers` and `peft`). It is a demonstration:
you are not expected to implement fine-tuning independently in this module. If you run it, compare your validation curve
with `FT["finetune"]["runs"]["lora"]["seeds"][0]["curve"]`: small differences are expected across machines and library
versions.